# GLOBAL CONSTANTS

In [ ]:
# Modify here
project_name = "NferenceInternalWorkingProject"
# project_name = "NSCLCWorkingProject"
version_of_work = "V8"
version_of_data = "V8"

## Don't modify below
basename_fetchdata_folder = f"/data/{project_name}/shared/Nbs_{version_of_work}/{version_of_data}/fetch_data/"
DATASET = "AMC2DataNSCLC"
TABLE_NAME  = f"stage_iii_nsclc_cohort_{version_of_work}"
SCRATCH_DB_NAME = "scratch_db." + TABLE_NAME
cohort_name_version_name_append = f"_{version_of_work}"
sys_path_of_cohortkit_package = f"/data/NSCLCWorkingProject/shared/Nbs_{version_of_work}/"
TABLE_NAME_2 = "study_co|hort_with_min_diagnosis_date"
SCRATCH_DB_NAME_WITH_STUDY_DTM = f"scratch_db.{TABLE_NAME_2}"

dicom_study_description_mapping_file_path = f'/data/NferenceInternalWorkingProject/shared/Nbs_{version_of_work}/{version_of_data}/input/Mayo_Radiology_Description_Mapping.tsv'

print(f"""
================ PROJECT CONFIGURATION ================

Project Name                     : {project_name}
Version of Work                  : {version_of_work}

Base Folder                      : {basename_fetchdata_folder}
Sys Path of cohortkit package    : {sys_path_of_cohortkit_package}

Dataset                          : {DATASET}
Table Name                       : {TABLE_NAME}
Scratch DB Name                  : {SCRATCH_DB_NAME}
Cohort Version Suffix            : {cohort_name_version_name_append}

Dicom study description mapping  : {dicom_study_description_mapping_file_path}


=======================================================
""")

In [ ]:
import logging

# Configure the root logger to the DEBUG level
logging.basicConfig(level=logging.INFO)

In [ ]:
import os
os.getenv('X_NFER_DATA')

In [ ]:
# os.environ['X_NFER_DATA'] = 'AMC1DataNSCLC'

In [ ]:
import nferhub
import pandas as pd
import numpy as np

cohort_client = nferhub.client("cohort", dataset=DATASET)

In [ ]:
sql_client = nferhub.client("sql_blaze")
sql_client.connect(dataset = "AMC2DataNSCLC",schema_model = "cdm",data_version = "4.015")

In [ ]:
import sys

# Add the exact parent directory containing the repository to Python's path
repo_path = "/data/NferenceInternalWorkingProject/shared/Nbs_V8/additional_nbs/Repos/nferHub"

if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

# Now run your import
import nferhub
print(nferhub.__file__)

In [ ]:
import nferhub
print(nferhub.__file__)

# Importing CSVs

In [ ]:
df = pd.read_csv("/data/NferenceInternalWorkingProject/shared/Nbs_V8/V8/process_data/RADIOLOGY_DICOM.csv", dtype=str)
duke_cohort_df =  pd.read_csv('/data/NferenceInternalWorkingProject/shared/Nbs_V8/additional_nbs/duke_cohort_20260616.csv')
image_availabilty_df = pd.read_csv('/data/NferenceInternalWorkingProject/shared/Nbs_V8/V8/fetch_data/image_availability.csv', dtype = str)

tagging_dataset_metadata_df = pd.read_csv('/data/NferenceInternalWorkingProject/shared/Nbs_V8/V8/process_data/TAGGING_DATASET_METADATA.csv')

In [ ]:
tagging_dataset_metadata_df.PERSON_ID.nunique()

# Creating Experiment

In [ ]:
# import os
# import tempfile
# import requests
# import pydicom
# import pandas as pd
# import time
# import nferhub

# # Initialize an AI Studio client
# aistudio_client = nferhub.client("ai-studio")

# # Creating Experiment and Dataset

# # Creating Experiment

# # Define a new Image Segmentation experiment.
# image_segmentation = aistudio_client.new_experiment(
#     data_type=aistudio_client.DataType.DicomSeries,
#     annotation_type=aistudio_client.AnnotationType.Segmentation,
# )


# # Define a new segmentation experiment.
# is_experiment = image_segmentation(
#     name="NSCLC_CCRT_RECIST_Tagging",
#     description="Annotate the region of tumor",
#     labels=["tumor"],
#     data_type=aistudio_client.DataType.DicomSeries,
# )

# # Push the experiment to AI Studio
# experiment = aistudio_client.create_experiment(is_experiment)
# print(experiment)

## Loading Existing Experiment

In [ ]:
import os
import tempfile
import requests
import pydicom
import pandas as pd
import time
import nferhub

# Initialize an AI Studio client

aistudio_client = nferhub.client("ai-studio")

experiment = aistudio_client.get_experiments()[0]
print(experiment)

In [ ]:
# experiment.add_owners([user])

In [ ]:
import pandas as pd
pd.options.display.max_rows = 200
pd.options.display.max_columns = None
pd.options.display.max_colwidth = None

## Duke Cohort

In [ ]:
duke_cohort_df.head()

In [ ]:
duke_cohort_df.info()

In [ ]:
duke_cohort_df.priority.value_counts(dropna = False)

In [ ]:
# duke_cohort_df = duke_cohort_df[duke_cohort_df.priority == True]

In [ ]:
duke_cohort_df.shape

In [ ]:
duke_cohort_df.drop(columns = ['rt_crt_start_date', 'chemo_crt_start_date',
                               'concur_durva_dt',  'subsq_durva_dt'], inplace = True )

In [ ]:
duke_cohort_df.rename(columns = {'person_id' : 'PERSON_ID'}, inplace = True)

In [ ]:
duke_cohort_df.head(2)

In [ ]:
randomized_df = (
    
    duke_cohort_df.sample(frac=1, random_state=42)        # shuffle randomly
    .sort_values('priority', ascending=False) # True first, False last
    .reset_index(drop=True)
)
randomized_df

In [ ]:
randomized_df.info()

## Radiology Dicom

In [ ]:
df.head()

In [ ]:
# df['STUDY_DATE'] = pd.to_datetime(df['STUDY_DATE'])
# df['EVENT_DATE'] = pd.to_datetime(df['EVENT_DATE'])

In [ ]:
# import pandas as pd

# # 1. Count how many cells are empty (NaN/Null) in each column
# empty_study = df['STUDY_DATE'].isna().sum()
# empty_event = df['EVENT_DATE'].isna().sum()

# # 2. Filter for rows where BOTH dates are actually present (not null)
# # This prevents NaN == NaN from accidentally being counted as a "same date"
# valid_dates_mask = df['STUDY_DATE'].notna() & df['EVENT_DATE'].notna()
# valid_df = df[valid_dates_mask]

# # 3. Count same vs different out of the valid dates
# # We use .astype(str) to ensure formatting mismatches don't mess up the comparison
# same_dates = (valid_df['STUDY_DATE'].astype(str) == valid_df['EVENT_DATE'].astype(str)).sum()
# different_dates = (valid_df['STUDY_DATE'].astype(str) != valid_df['EVENT_DATE'].astype(str)).sum()

# # 4. Print the breakdown beautifully
# print("--- Date Comparison Breakdown ---")
# print(f"Total Rows in DataFrame : {len(df)}")
# print(f"Same Dates              : {same_dates}")
# print(f"Different Dates         : {different_dates}")
# print(f"Empty in STUDY_DATE     : {empty_study}")
# print(f"Empty in EVENT_DATE     : {empty_event}")

In [ ]:
df.PERSON_ID.nunique()

In [ ]:
df.info()

In [ ]:
df = df[['MODALITY_DESCRIPTION' , 'STUDY_DATE' , 'PERSON_ID' , 'STUDY_INSTANCE_UID' ,
         'SERIES_INSTANCE_UID', 'HARMONIZED_STUDY_DESCRIPTION']]
df.head()

In [ ]:
df['MODALITY_DESCRIPTION'].value_counts(dropna = False)

In [ ]:
filter_list = ['CT' ,'PT', 'MR']
df = df[df['MODALITY_DESCRIPTION'].isin(filter_list)]

#### sampling

In [ ]:
#random_id = df['PERSON_ID'].drop_duplicates().sample(n=1)
sampled_df = df[df['PERSON_ID'] == '3692632']

In [ ]:
sampled_df.shape

In [ ]:
sampled_df.rename(columns = { 'SERIES_INSTANCE_UID': 'series_id' , 'STUDY_INSTANCE_UID': 'study_id' ,
                              'MODALITY_DESCRIPTION': 'modality' , 'PERSON_ID': 'nfer_pid' } , inplace = True)

In [ ]:
sampled_df.sort_values(by = ['STUDY_DATE'], inplace = True)

In [ ]:
sampled_df.head(10)

In [ ]:
sampled_df.modality.value_counts(dropna = False)

In [ ]:
sampled_df.head()

In [ ]:
sampled_df.rename(columns = {'nfer_pid': 'PERSON_ID'}, inplace = True)

In [ ]:
sampled_df['PERSON_ID'] = sampled_df['PERSON_ID'].astype(int)

In [ ]:
randomized_df.info()

In [ ]:
import pandas as pd

def categorize_scan_dates(sampled_df, randomized_df):
    """
    Merges sampled_df with mayo_cohort_df and categorizes EVENT_DATEs into
    BASELINE_SCAN_DATES and FOLLOWUP_SCAN_DATES per patient.
    
    Baseline:  EVENT_DATE >= (crt_start_date - 90 days) AND EVENT_DATE < crt_start_date
    Followup:  EVENT_DATE >= crt_end_date
    Filter:    Only EVENT_DATEs >= 2018-01-01
    """
    
    # --- 1. Filter events from 2018 onwards ---
    sampled_df = sampled_df.copy()
    sampled_df['STUDY_DATE'] = pd.to_datetime(sampled_df['STUDY_DATE'])
    sampled_df = sampled_df[sampled_df['STUDY_DATE'] >= '2018-01-01'].copy()
    
    # --- 2. Prepare mayo_cohort_df ---
    random = randomized_df[['PERSON_ID', 'study_index_date', 'crt_start_date', 
                            'crt_end_date', 'overall_stage', 
                            't_value', 'n_value', 'm_value']].copy()
    
    random['crt_start_date'] = pd.to_datetime(random['crt_start_date'])
    random['crt_end_date']   = pd.to_datetime(random['crt_end_date'])
    random['study_index_date'] = pd.to_datetime(random['study_index_date'])
    
    # --- 3. Merge on patient ID ---
    merged = sampled_df.merge(
        random,
        
        on='PERSON_ID',
        how='inner'
    )
    
    # --- 4. Compute baseline window: [crt_start_date - 90 days, crt_start_date) ---
    merged['baseline_start'] = merged['crt_start_date'] - pd.Timedelta(days=90)
    
    baseline_mask = (
        (merged['STUDY_DATE'] >= merged['baseline_start']) &
        (merged['STUDY_DATE'] < merged['crt_start_date'])
    )
    
    followup_mask = merged['STUDY_DATE'] >= merged['crt_end_date']
    
    merged['_baseline_date'] = merged['STUDY_DATE'].where(baseline_mask)
    merged['_followup_date'] = merged['STUDY_DATE'].where(followup_mask)
    
    # --- 5. Aggregate per patient ---
    def collect_dates(series):
        """Return sorted list of unique non-null dates as date strings."""
        return sorted(set(
            d.date().isoformat() 
            for d in series.dropna()
        ))
    
    result = (
        merged.groupby('PERSON_ID', as_index=False)
        .agg(
            study_index_date = ('study_index_date', 'first'),
            crt_start_date   = ('crt_start_date',   'first'),
            crt_end_date     = ('crt_end_date',     'first'),
            overall_stage    = ('overall_stage',    'first'),
            t_value          = ('t_value',          'first'),
            n_value          = ('n_value',          'first'),
            m_value          = ('m_value',          'first'),
            BASELINE_SCAN_DATES = ('_baseline_date', collect_dates),
            FOLLOWUP_SCAN_DATES = ('_followup_date', collect_dates),
        )
    )
    
    # --- 6. Clean up date formatting ---
    for col in ['study_index_date', 'crt_start_date', 'crt_end_date']:
        result[col] = pd.to_datetime(result[col]).dt.date.astype(str)
    
    return result



sample_detail_df = categorize_scan_dates(sampled_df, randomized_df)
sample_detail_df.head()

## Image Availabilty( De - Identified Images of Patients)

In [ ]:
image_availabilty_df

In [ ]:
image_availabilty_df.rename(columns = { 'SERIES_INSTANCE_UID': 'series_id' , 'STUDY_INSTANCE_UID': 'study_id' ,
                              'SOURCE_MODALITY': 'modality' , 'PERSON_ID': 'nfer_pid' } , inplace = True)

In [ ]:
sampled_df.rename(columns = {  'PERSON_ID': 'nfer_pid' } , inplace = True)

In [ ]:
# sampled_df.rename(columns = {'nfer_pid' : 'PERSON_ID'}, inplace = True)
image_availabilty_df['nfer_pid'] = image_availabilty_df['nfer_pid'].astype(str)
sampled_df['nfer_pid'] = sampled_df['nfer_pid'].astype(str)

In [ ]:
final_sample_df = pd.merge(sampled_df , image_availabilty_df,
                           on = ['nfer_pid', 'study_id', 'series_id', 'modality' ] , how= 'inner')

In [ ]:
final_sample_df.shape

In [ ]:
# final_sample_df.modality.value_counts(dropna = False)

In [ ]:
sampled_df['STUDY_DATE'] = pd.to_datetime(sampled_df['STUDY_DATE'], errors = 'coerce')

In [ ]:
sampled_df = sampled_df[sampled_df['STUDY_DATE'].dt.year >=2018]

In [ ]:
sampled_df.head(10)

In [ ]:
sampled_df.rename(columns = {'PERSON_ID': 'nfer_pid'}, inplace =True)

In [ ]:
pid= sampled_df['nfer_pid'].drop_duplicates().astype(int)
pid

In [ ]:
# sampled_df['nfer_pid'] = sampled_df['nfer_pid'].astype(int)

In [ ]:
# import numpy as np
# import pandas as pd
# import ast

# def get_patients_with_active_dates(df, col1, col2, pid_col="PERSON_ID"):
#     """
#     Returns a list of unique patient IDs where both specified date columns
#     contain non-empty lists, handling unexpected nested arrays safely.
#     """
    
#     def safe_parse_list(val):
#         # 1. If it's a numpy array, list, or series, check if it's empty
#         if isinstance(val, (list, np.ndarray, pd.Series)):
#             return list(val) if len(val) > 0 else []
            
#         # 2. Check for scalar NaN safely using a try/except or isinstance check
#         if isinstance(val, float) and np.isnan(val):
#             return []
            
#         # 3. If it's a fallback or not a string, return empty
#         if pd.isna(val) if not isinstance(val, (list, np.ndarray, pd.Series)) else False or not isinstance(val, str):
#             # Let's simplify the safety check to avoid any scalar/array confusion:
#             pass
            
#         # Refactored bulletproof safety block:
#         if not isinstance(val, str):
#             return []
            
#         try:
#             parsed = ast.literal_eval(val)
#             return parsed if isinstance(parsed, list) else []
#         except (ValueError, SyntaxError):
#             return []

#     # Better, cleaner implementation of the internal helper:
#     def safe_parse_list_clean(val):
#         # If it's already an array/list object
#         if isinstance(val, (list, np.ndarray, pd.Series)):
#             return list(val) if len(val) > 0 else []
#         # If it's missing/null/NaN
#         if pd.isna(val) is True: 
#             return []
#         # If it isn't a string, we can't parse it with literal_eval
#         if not isinstance(val, str):
#             return []
            
#         try:
#             parsed = ast.literal_eval(val)
#             return parsed if isinstance(parsed, list) else []
#         except (ValueError, SyntaxError):
#             return []

#     # Apply the clean helper
#     list1 = df[col1].apply(safe_parse_list_clean)
#     list2 = df[col2].apply(safe_parse_list_clean)

#     condition = (list1.str.len() > 0) & (list2.str.len() > 0)
#     matching_pids = df.loc[condition, pid_col].unique().tolist()

#     return matching_pids

In [ ]:
# required_pids = get_patients_with_active_dates(final_merged_df, col1 = 'LIST_OF_STUDY_DATES_OF_RADIOLOGY_SCANS_BASELINE',
#                                            col2 = 'LIST_OF_STUDY_DATES_OF_RADIOLOGY_SCANS_FOLLOWUP', pid_col="nfer_pid")

In [ ]:
# required_pids = pd.DataFrame(required_pids)

In [ ]:
def generate_dataset_description(row):

    def format_val(val):
        if isinstance(val, list):
            return ", ".join(str(v) for v in val) if val else ""
        elif pd.isna(val):
            return ""
        else:
            return str(val)

    def format_tnm(val):
        try:
            if pd.isna(val):
                return ""
        except (TypeError, ValueError):
            pass
        return str(val) if val not in [None, "", "nan", "NaN"] else ""

    description = f"""STUDY_INDEX_DATE: {format_val(row['study_index_date'])} ;
overall_stage: {format_val(row['overall_stage'])} ;
T_VALUE: {format_tnm(row['t_value'])} ;
N_VALUE: {format_tnm(row['n_value'])} ;
M_VALUE: {format_tnm(row['m_value'])} ;
CCRT_START_DATE: {format_val(row['crt_start_date'])} ;
CCRT_END_DATE: {format_val(row['crt_end_date'])} ;
Baseline scan dates: {format_val(row['BASELINE_SCAN_DATES'])} ;
Follow-Up scan dates: {format_val(row['FOLLOWUP_SCAN_DATES'])}"""

    return description

In [ ]:
patient_row = sample_detail_df.iloc[0]

In [ ]:
dynamic_description = generate_dataset_description(patient_row)

### Uploading Image Data

In [ ]:
records2 = []
counter = 0
for _, row in final_sample_df.iterrows(): 
    records2.append({
                "modality": row.modality,
                "patient_id": row.nfer_pid,
                "study_uid": row.study_id,
                "series_uid": row.series_id,
                "instance_uid": row.get('instance_id', ''),
            })
    
        
dataset_data = {"images": records2}


In [ ]:
len(records2)

In [ ]:
records3 = [
    {str(key): str(value) for key, value in x.items()} 
    for x in records2
]

In [ ]:
# for x in records3:
#     for y , z in x.items():
#         print(y,z)
#         print(type(y))
#         print(type(z))

dataset_data = {"images": records3}
len(records3)

In [ ]:
dataset_data

In [ ]:
time.sleep(30)

In [ ]:
dataset = experiment.create_dataset(
    name="dataset_2_3692632",  # manually changing the name of dataset
    description=dynamic_description,
    json_data=dataset_data
)

print(dataset)

In [ ]:
# List of emails you want to search for
emails = [
    # 'Sangeetha.p@nference.net'
    # 'anila.mathew.nference@nfer-workspaces.com',
    # 'pkumar.nference@nfer-workspaces.com',
    # 'Sangeetha.p.nference@nfer-workspaces.com',
    # 'mohammed.imran.nference@nfer-workspaces.com' 
  
]

user_list = []

for email in emails:
    try:
        # Search for each email individually
        result = aistudio_client.search_users(email)
        user_list.extend(result)
    except Exception as e:
        print(f"Could not find user: {email}")

print(user_list)

tagger = user_list[:]
dataset.add_taggers(taggers=tagger)
print(dataset)

### Exporting annotated data

#### Helper method

In [ ]:
experiment_name = "NSCLC_CCRT_RECIST_Tagging"
dataset_name = "dataset_6_93207188"
tagger_name = "mohammed.imran.nference@nfer-workspaces.com"

In [ ]:
tmp_dir = tempfile.gettempdir()
def download_file(url: str):
    file_name = url.split("/")[-1]
    file_path = os.path.join(tmp_dir, file_name)
    with requests.get(f"{os.getenv('X_NFER_BASEURL').strip('/')}/{url.strip('/')}", 
                  auth=(os.getenv('WORKSPACE_USER'), os.getenv('WORKSPACE_USER_TOKEN'))) as response:

        if response.status_code == 200:
            with open(file_path, "wb") as f:
                f.write(response.content)

                return file_path

        else:
            print(f"Failed to fetch blob. Status code: {response.status_code}")
                

#### Download SR DCMS

In [ ]:
def download_SR_for_dataset(experiment_name, dataset_name="", tagger_name="") -> list[str]:
    
    ### exporting dataset
    
    experiment = aistudio_client.get_experiments(query=experiment_name)[0]
    # print(experiment)

    # List datasets in the experiment.
    datasets = experiment.get_datasets(query=dataset_name)
    # [print(dataset) for dataset in datasets]

    # Export the samples with annotation
    exported_data = experiment.export_dataset(dataset_id=datasets[0].id, tagger=[tagger_name], transform=False)
    # print(exported_data)

    #print(f"got {len(exported_data['images'])} samples in the dataset")
    
    ### downloading dataset SRs to tmp dir
    SR_list = []
    for sample in exported_data.get('images', []):
        SR_url = sample.get('tag', {}).get('annotation_url', None)
        if SR_url is not None:
            SR_file_path = download_file(SR_url)
            SR_list.append((SR_file_path, tagger_name))
            
    return SR_list

annotations = download_SR_for_dataset(experiment_name, dataset_name, tagger_name)
print(annotations)

#### Creating CSV for annotations

In [ ]:
# import ast
# import pydicom
# import pandas as pd
# import os
# import logging
# import numpy as np

# # --- LOGGING SETUP ---
# logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
# logger = logging.getLogger(__name__)

# def find_coordinates(item, coord_dict):
#     """Recursively searches through an SR content item to find Graphic Data."""
#     val_type = getattr(item, 'ValueType', None)
#     if val_type == "SCOORD" and 0x00700022 in item:
#         coord_dict['Pixel'] = str(list(item[0x00700022].value))
#     elif val_type == "SCOORD3D" and 0x00700022 in item:
#         coord_dict['Physical'] = str(list(item[0x00700022].value))
        
#     if 0x0040A730 in item:
#         for child in item[0x0040A730]:
#             find_coordinates(child, coord_dict)

# def find_referenced_sop_uid(item):
#     """
#     Robust helper to traverse finding the ReferencedSOPInstanceUID
#     without relying on hardcoded content positions.
#     """
#     if hasattr(item, 'ReferencedSOPInstanceUID'):
#         return item.ReferencedSOPInstanceUID
        
#     if 0x00081199 in item:  # Referenced SOP Sequence
#         for sub_item in item[0x00081199]:
#             if hasattr(sub_item, 'ReferencedSOPInstanceUID'):
#                 return sub_item.ReferencedSOPInstanceUID

#     if 0x0040A730 in item:  # Content Sequence recursion
#         for child in item[0x0040A730]:
#             res = find_referenced_sop_uid(child)
#             if res:
#                 return res
#     return None

# def create_csv_for_annotations(SR_list):
#     measurements = []
#     patientID = "Unknown"

#     for SR_record in SR_list:
#         try:
#             SR_file, tagger = SR_record
#             ds = pydicom.dcmread(SR_file)
#             patientID = getattr(ds, 'PatientID', "Unknown")
            
#             # --- STUDY DATE ---
#             study_date = ds.get(0x00080020).value if ds.get(0x00080020) else "Unknown"
            
#             # --- MODALITY ---
#             modality = getattr(ds, 'Modality', "Unknown")
            
#             # --- SERIES DESCRIPTION CLEANING ---
#             full_desc = getattr(ds, 'SeriesDescription', "Unknown")
#             description = " ".join(full_desc.split(' ')[-2:]) if " " in full_desc else full_desc
            
#             # --- INSTANCE MAPPING ---
#             instanceMap = {}
#             if 0x0040A375 in ds:
#                 for ref_study in ds[0x0040A375]:
#                     s_uid = getattr(ref_study, 'StudyInstanceUID', "Unknown")
#                     if 0x00081115 in ref_study:
#                         for ref_series in ref_study[0x00081115]:
#                             se_uid = getattr(ref_series, 'SeriesInstanceUID', "Unknown")
#                             if 0x00081199 in ref_series:
#                                 for ref_inst in ref_series[0x00081199]:
#                                     i_uid = getattr(ref_inst, 'ReferencedSOPInstanceUID', "Unknown")
#                                     instanceMap[i_uid] = (s_uid, se_uid)

#             # --- PROCESS MEASUREMENTS ---
#             if 0x0040A730 in ds:
#                 for container in ds[0x0040A730]:
#                     if 0x0040A730 not in container: continue
#                     for measurement in container[0x0040A730]:
                        
#                         # Gather coordinates first to determine if this item has active data
#                         coords = {'Pixel': "N/A", 'Physical': "N/A"}
#                         find_coordinates(measurement, coords)
                        
#                         # Grab the Instance UID dynamically
#                         inst_uid = find_referenced_sop_uid(measurement)
                        
#                         # --- CRITICAL FILTER ---
#                         # Skip container metadata blocks that lack valid coordinates or target image references
#                         if coords['Pixel'] == "N/A" or not inst_uid or inst_uid == "Unknown":
#                             continue

#                         m_obj = {
#                             "PatientID": patientID,
#                             "StudyDate": study_date,
#                             "Modality": modality,
#                             "descriptions": description,
#                             "Tagger": tagger,
#                             "PixelCoordinates": coords['Pixel'],
#                             "PhysicalCoordinates": coords['Physical'],
#                             "InstanceUID": inst_uid,
#                             "StudyUID": "Unknown",
#                             "SeriesUID": "Unknown"
#                         }
                        
#                         m_obj["StudyUID"], m_obj["SeriesUID"] = instanceMap.get(inst_uid, ("Unknown", "Unknown"))
                        
#                         # RECIST Metadata extraction safely via loop matching or soft indexes
#                         try:
#                             m_content = measurement[0x0040A730]
#                             try:
#                                 m_obj["Unit"] = m_content[3][0x0040A300][0][0x004008EA][0][0x00080100].value
#                             except: pass
#                             try:
#                                 m_obj["Label"] = m_content[2][0x0040A168][0][0x00080104].value
#                             except: pass
#                             try:
#                                 m_obj["Length"] = m_content[3][0x0040A300][0][0x0040A30A].value
#                             except: pass
#                             try:
#                                 m_obj["Width"] = m_content[4][0x0040A300][0][0x0040A30A].value
#                             except: pass
#                         except Exception:
#                             pass
                        
#                         measurements.append(m_obj)
                        
#         except Exception as e:
#             logger.error(f"Error in {SR_record[0]}: {e}")

#     if measurements:
#         HEADERS = [
#             "PatientID", "StudyDate", "StudyUID", "SeriesUID", "InstanceUID",
#             "Modality", "descriptions", "Label", "Length", "Width",
#             "Unit", "PixelCoordinates", "PhysicalCoordinates", "Tagger"
#         ]
#         df = pd.DataFrame(measurements, columns=HEADERS)
        
#         # Drop duplicates across the critical metadata tracks just in case software logged duplicate nodes
#         df.drop_duplicates(subset=["InstanceUID", "PixelCoordinates", "descriptions"], keep="last", inplace=True)
        
#         # Standardize Study Date
#         df['StudyDate'] = pd.to_datetime(df['StudyDate'], format='%Y%m%d', errors='coerce').dt.date
        
#         # Floor truncation mapping for numeric values
#         if 'Length' in df.columns and 'Width' in df.columns:
#             df[['Length', 'Width']] = df[['Length', 'Width']].apply(pd.to_numeric, errors='coerce')
#             df[['Length', 'Width']] = np.floor(df[['Length', 'Width']] * 10000) / 10000
            
#         # Integer Pixel Coordinates Processing
#         df['PixelCoordinates'] = df['PixelCoordinates'].apply(
#             lambda x: [int(round(float(p))) for p in ast.literal_eval(x)] if x != "N/A" and pd.notna(x) else x
#         )
        
#         # Rename column as required
#         df.rename(columns={'PatientID': 'PERSON_ID'}, inplace=True)
        
#         fname = f"{patientID}.csv"
#         df.to_csv(fname, mode="a" if os.path.exists(fname) else "w", header=not os.path.exists(fname), index=False)
#         logger.info(f"Saved {len(df)} unique clean rows to {fname}")

# if __name__ == "__main__":
#     # Ensure 'annotations' list is populated by your download function
#     create_csv_for_annotations(annotations)

# Exploration

In [ ]:
import nferhub

In [ ]:
sql_client = nferhub.client("sql_blaze")
sql_client.connect(dataset = "AMC2DataNSCLC",schema_model = "cdm",data_version = "4.015")

In [ ]:
t = sql_client.query(''' select distinct ia.STUDY_INSTANCE_UID , ia.SERIES_INSTANCE_UID  from IMAGE_AVAILABILITY as ia

where ia.PERSON_ID == 3692632
''') 
t

In [ ]:
t = sql_client.query(''' select count(*)  from RADIOLOGY_DICOM_V2 as rd 
INNER JOIN DIM_SYN_MODALITY as dm
    ON dm.SOURCE_CONCEPT_ID = rd.MODALITY_SOURCE_CONCEPT_ID
where rd.PERSON_ID == 3692632
and dm.SOURCE_DESCRIPTION in ['MR', 'CT', 'PT'] ''') 
t